# 01 Exploration

Explorative Sicht auf die aufbereiteten Daten. Die Pipeline (`afe.bat all`) bleibt die verbindliche Auswertung; hier wird nur geschaut, nicht bereinigt.

**Vor dem Commit Outputs löschen:** `afe.bat strip` (öffentliches Repo).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from afe.pipeline import build

pd.set_option('display.max_columns', 50)
ds = build(write=False)
ds.flow

## Besuche (Analysestichprobe)

In [ ]:
cols = ['idvisit', 'start', 'variant', 'n_pageviews', 'time_total_s', 'time_to_form_s', 'n_clicks', 'dir_change_rate', 'aoi_portrait_dwell_ms']
ds.included[cols].sort_values('start')

## Zeitverlauf eines Besuchs

Maus-, Klick- und Scrollereignisse über die Zeit. `VISIT` anpassen.

In [ ]:
VISIT = int(ds.included['idvisit'].iloc[0])
ev = ds.events[ds.events['idvisit'] == VISIT]
pv = ds.pageviews[ds.pageviews['idvisit'] == VISIT]
offset = dict(zip(pv['idloghsr'], pv['time_on_page_ms'].cumsum().shift(fill_value=0)))
t = (ev['idloghsr'].map(offset) + ev['t_ms']) / 1000
fig, ax = plt.subplots(figsize=(10, 2.5))
for name, y in [('move', 0), ('click', 1), ('scroll', 2), ('form_value', 3), ('form_text', 3)]:
    m = ev['event'] == name
    ax.scatter(t[m], [y] * m.sum(), s=6, label=name)
ax.set_yticks([0, 1, 2, 3], ['Bewegung', 'Klick', 'Scroll', 'Formular'])
ax.set_xlabel('Sekunden (aktive Zeit)')
ax.set_title(f'Besuch {VISIT} ({ds.visits.set_index("idvisit").loc[VISIT, "variant"]})')
plt.show()

## Meistbenutzte Elemente pro Seite

In [ ]:
from afe.features import selector_summary
selector_summary(ds.events[ds.events['included'].fillna(False).astype(bool)]).groupby('page').head(5)